In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('processed_data.csv', parse_dates=['datetime'])

print(f" Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(df.head(3))

In [ ]:
numeric_cols = [
    'Global_active_power', 'Global_reactive_power',
    'Voltage', 'Global_intensity',
    'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]

datetime_col = df['datetime']

# Check for NaNs
print(" NaN counts per column:")
print(df[numeric_cols].isnull().sum())

# Drop rows with any NaN
df_clean = df[numeric_cols].dropna()
datetime_col = df['datetime'][df[numeric_cols].notna().all(axis=1)].reset_index(drop=True)
df_clean = df_clean.reset_index(drop=True)

print(f"\n Rows after dropping NaNs: {len(df_clean):,}")

# Check std
print("\n Std per column:")
print(df_clean.std().round(4))

# Extract values
data = df_clean.values

col_means = data.mean(axis=0)
col_stds  = data.std(axis=0)
col_mins  = data.min(axis=0)
col_maxs  = data.max(axis=0)

# Add tiny noise to prevent singular matrix
data_for_corr = data + np.random.normal(0, 1e-9, data.shape)

# Correlation matrix
corr_matrix = np.corrcoef(data_for_corr, rowvar=False)

# Make matrix positive definite
corr_matrix = (corr_matrix + corr_matrix.T) / 2
min_eig = np.linalg.eigvalsh(corr_matrix).min()
if min_eig < 0:
    corr_matrix -= 1.1 * min_eig * np.eye(corr_matrix.shape[0])
    print(" Correlation matrix adjusted to be positive definite")

# Cholesky decomposition
cholesky_L = np.linalg.cholesky(corr_matrix)

print("\n Stats computed successfully!")
print("\nCorrelation matrix:")
print(pd.DataFrame(corr_matrix, columns=numeric_cols, index=numeric_cols).round(2))

In [ ]:
import os

os.makedirs("/content/synthetic_datasets", exist_ok=True)

NOISE_SCALE = 0.05
N      = len(data)
n_cols = len(numeric_cols)

for i in range(1, 50):
    raw_noise        = np.random.randn(N, n_cols)
    correlated_noise = raw_noise @ cholesky_L.T
    scaled_noise     = correlated_noise * (col_stds * NOISE_SCALE)
    synthetic_data   = np.clip(data + scaled_noise, col_mins, col_maxs)

    synth_df = pd.DataFrame(synthetic_data, columns=numeric_cols)
    synth_df.insert(0, 'datetime', datetime_col.values)
    synth_df.to_csv(f"/content/synthetic_datasets/synthetic_{i:02d}.csv", index=False)

    if i % 10 == 0 or i == 1:
        print(f"  {i}/49 done...")

print("\n 49 datasets generated!")

In [ ]:
comparison = pd.DataFrame({
    'Real Mean':  col_means.round(4),
    'Synth Mean': synth_df[numeric_cols].mean().values.round(4),
    'Real Std':   col_stds.round(4),
    'Synth Std':  synth_df[numeric_cols].std().values.round(4),
    'Real Min':   col_mins.round(4),
    'Synth Min':  synth_df[numeric_cols].min().values.round(4),
    'Real Max':   col_maxs.round(4),
    'Synth Max':  synth_df[numeric_cols].max().values.round(4),
}, index=numeric_cols)

print(comparison)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/all_synthetic_datasets", 'zip', "/content/synthetic_datasets")
files.download("/content/all_synthetic_datasets.zip")

print("Download started!")